<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-3-ai-agents/lab-01-minimal-agent-for-kestrel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 (graded) — Minimal agent for Kestrel
**Course 3: AI Agents and Agentic AI with Python — Chapter 1: The agent loop**

**Problem brief (Leo Farkas, Kestrel Logistics):** "Our ops chatbot can't answer 'what's the
driving time from the depot to the port right now' — it needs to look things up. But I've
been burned by 'AI agents' that spin forever. Build the simplest thing that works, and show
me the guardrails."

**What you'll submit:** the from-scratch ReAct loop with two real tools, all four
guardrails demonstrated firing on purpose, and the "agent vs workflow" note.

## 1. Two real tools

In [ ]:
import urllib.request, json, urllib.parse

MOCK_COORDS = {'kestrel depot': {'lat': 41.85, 'lon': -87.65}, 'port authority': {'lat': 40.7, 'lon': -74.0}}

def geocode(place):
    # Kestrel Depot / Port Authority are fictional course-cast places, so a real geocoder
    # legitimately returns zero results for them (not an error) — same offline fallback
    # covers that case and any real network/API failure.
    try:
        url = f'https://nominatim.openstreetmap.org/search?q={urllib.parse.quote(place)}&format=json&limit=1'
        req = urllib.request.Request(url, headers={'User-Agent': 'aibits-course-lab/1.0'})
        with urllib.request.urlopen(req, timeout=5) as resp:
            results = json.load(resp)
        if results:
            return {'lat': float(results[0]['lat']), 'lon': float(results[0]['lon'])}
        raise ValueError('place not found by the real geocoder')
    except Exception as e:
        return MOCK_COORDS.get(place.lower(), {'lat': 0.0, 'lon': 0.0, 'note': f'mock (real API failed: {e})'})

def driving_distance_km(coord_a, coord_b):
    """Haversine great-circle distance as a real-tool stand-in for a routing API (no key needed)."""
    import math
    R = 6371
    lat1, lon1, lat2, lon2 = map(math.radians, [coord_a['lat'], coord_a['lon'], coord_b['lat'], coord_b['lon']])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = math.sin(dlat/2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon/2)**2
    return round(R * 2 * math.asin(math.sqrt(a)) * 1.3, 1)  # *1.3 as a rough road-vs-straight-line factor

print(geocode('Kestrel Depot'))

## 2. The tool registry

In [ ]:
def tool_geocode(place: str) -> dict:
    """Look up the latitude/longitude of a named place."""
    return geocode(place)

def tool_distance(place_a: str, place_b: str) -> dict:
    """Estimate the driving distance in km between two named places."""
    a, b = geocode(place_a), geocode(place_b)
    if 'error' in a or 'error' in b:
        return {'error': 'could not geocode one of the places'}
    return {'distance_km': driving_distance_km(a, b)}

def tool_final_answer(text: str) -> dict:
    """Return the final answer to the user and stop the loop."""
    return {'final': text}

TOOLS = {'geocode': tool_geocode, 'distance': tool_distance, 'final_answer': tool_final_answer}

## 3. The agent's "brain": hosted LLM, or a scripted mock
Fill in the `TODO` for the hosted path's tool-calling parse. The mock path is a scripted
policy that inspects the history so far — it demonstrates the exact same loop mechanics with
zero cost and zero key.

In [ ]:
import os, json as _json
try:
    from google.colab import userdata
    API_KEY = userdata.get('LLM_API_KEY'); BASE_URL = userdata.get('LLM_BASE_URL')
except Exception:
    API_KEY = os.environ.get('LLM_API_KEY'); BASE_URL = os.environ.get('LLM_BASE_URL')
hosted_available = bool(API_KEY and BASE_URL)

TOOL_SCHEMAS = [
    {'type': 'function', 'function': {'name': 'geocode', 'description': 'Look up lat/lon of a place.',
        'parameters': {'type': 'object', 'properties': {'place': {'type': 'string'}}, 'required': ['place']}}},
    {'type': 'function', 'function': {'name': 'distance', 'description': 'Estimate driving distance in km.',
        'parameters': {'type': 'object', 'properties': {'place_a': {'type': 'string'}, 'place_b': {'type': 'string'}}, 'required': ['place_a', 'place_b']}}},
    {'type': 'function', 'function': {'name': 'final_answer', 'description': 'Give the final answer and stop.',
        'parameters': {'type': 'object', 'properties': {'text': {'type': 'string'}}, 'required': ['text']}}},
]

def think_hosted(history, goal):
    from openai import OpenAI
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    messages = [{'role': 'system', 'content': 'You are an ops assistant. Use tools to answer, then call final_answer.'}]
    messages.append({'role': 'user', 'content': goal})
    for h in history:
        messages.append({'role': 'assistant', 'content': None, 'tool_calls': [h['tool_call']]})
        messages.append({'role': 'tool', 'tool_call_id': h['tool_call']['id'], 'content': _json.dumps(h['result'])})
    resp = client.chat.completions.create(model='llama-3.1-8b-instant', messages=messages, tools=TOOL_SCHEMAS, max_tokens=200)
    # TODO: extract the tool name + arguments from resp.choices[0].message.tool_calls[0]
    call = resp.choices[0].message.tool_calls[0]
    return call.function.name, _json.loads(call.function.arguments), call.id

def think_mock(history, goal):
    """A scripted policy for the offline path: geocode both places, get the distance, answer."""
    steps_done = [h['tool_name'] for h in history]
    if 'distance' not in steps_done:
        return 'distance', {'place_a': 'Kestrel Depot', 'place_b': 'Port Authority'}, 'mock-1'
    dist = next(h['result']['distance_km'] for h in history if h['tool_name'] == 'distance')
    return 'final_answer', {'text': f'The driving distance from the depot to the port is approximately {dist} km.'}, 'mock-2'

def think(history, goal):
    return think_hosted(history, goal) if hosted_available else think_mock(history, goal)

## 4. The loop, with all four guardrails

In [ ]:
import time

def run_agent(goal, max_steps=6, timeout_s=30, max_cost_units=10, think_fn=None):
    think_fn = think_fn or think  # overridable, so Lab section 5 can inject a "bad" brain cleanly
    start = time.time()
    history = []
    cost_spent = 0
    for step in range(max_steps):
        if time.time() - start > timeout_s:
            return {'status': 'timeout', 'history': history}
        if cost_spent >= max_cost_units:
            return {'status': 'budget_exceeded', 'history': history}

        tool_name, args, call_id = think_fn(history, goal)
        cost_spent += 1  # one "unit" per LLM call — a real system would track actual tokens

        if tool_name not in TOOLS:
            return {'status': 'explicit_failure', 'reason': f'hallucinated tool: {tool_name}', 'history': history}

        result = TOOLS[tool_name](**args)
        history.append({'step': step, 'tool_name': tool_name, 'args': args, 'result': result,
                          'tool_call': {'id': call_id, 'type': 'function',
                                        'function': {'name': tool_name, 'arguments': _json.dumps(args)}}})

        if tool_name == 'final_answer':
            return {'status': 'done', 'answer': result['final'], 'history': history}

    return {'status': 'max_steps_reached', 'history': history}  # explicit failure, never a silent guess

result = run_agent('What is the driving time — well, distance — from the depot to the port right now?')
print('status:', result['status'])
if result['status'] == 'done':
    print('answer:', result['answer'])
for h in result['history']:
    print(' ', h['tool_name'], h['args'], '->', h['result'])

## 5. Demonstrate all four guardrails firing on purpose

In [ ]:
# 1. Max iteration count
r1 = run_agent('An impossible goal that never resolves', max_steps=1)
print('1. max_steps:', r1['status'])

# 2. Wall-clock timeout
r2 = run_agent('goal', timeout_s=0.0)
print('2. timeout:', r2['status'])

# 3. Cost/token budget
r3 = run_agent('goal', max_cost_units=0)
print('3. budget:', r3['status'])

# 4. Hallucinated tool call -> explicit failure, not a crash or a silent guess
def think_bad(history, goal):
    return 'delete_everything', {}, 'bad-1'

r4 = run_agent('goal', think_fn=think_bad)
print('4. hallucinated tool ->', r4['status'], '-', r4.get('reason'))

## 6. Agent vs. workflow (fill in)
Kestrel's actual request — geocode two places, compute a distance, report it — has a known,
fixed sequence of steps. Was building an agent (vs. a 3-step fixed workflow) the right call
here? Write the honest answer, including if it's "no."

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 1: The agent loop*